# Lab 2: ML Deployment for LLM-Based Ticket Routing

In this lab, you’ll continue from where Lab 1 ended. You’ve trained and validated your models. Now you’ll:
1. Package your selected model for deployment
2. Set up a REST API for prediction
3. Simulate production monitoring and document governance

We'll cover the remaining three stages of the ML lifecycle:
- Packaging & Continuous Delivery (CD)
- Production Monitoring & Observability
- Governance & Continuous Improvement

**Make sure to run Steps 0-3 from the previous lab first.**

## Step 0: Environment Setup and Data Ingestion

In [ ]:
!pip install pandas scikit-learn sentence-transformers mlflow matplotlib seaborn

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import json
import os

'pip' is not recognized as an internal or external command,
operable program or batch file.


## Step 1: Data Intake & Feature Management

In [ ]:
# Data

noisy_samples = [
    ("Login problem, can't reach my account", "Technical Support"),
    ("can't log in lol help", "Technical Support"),
    ("Need help wit billing and login", "Account Management"),
    ("Charged twice!!! Refund pls", "Billing"),
    ("Whre is my invoice", "Billing"),
    ("How to chnge plan", "Account Management"),
    ("i think i want to cancel maybe", "Account Management"),
    ("Why am I being charged so much?", "Billing"),
    ("crashes on load up", "Technical Support"),
    ("can't reset pass", "Technical Support"),
    ("want refund and cancel", "Billing"),
    ("need help – can't access app or billing", "Technical Support"),
    ("pls fix app it stuck", "Technical Support"),
    ("is my data safe?", "General Inquiry"),
    ("do u have support chat?", "General Inquiry"),
    ("invoice not showing", "Billing"),
    ("how to export all stuff?", "Account Management"),
    ("upgrade plan to annual", "Account Management"),
    ("app broken again sigh", "Technical Support"),
    ("change payment info how", "Billing"),
    ("i emailed but no response", "General Inquiry"),
    ("just want to close account fast", "Account Management"),
    ("overcharged for month i think?", "Billing"),
    ("subscription disappeared", "Technical Support"),
    ("does billing work weekends", "General Inquiry"),
    ("add new team member pls", "Account Management"),
    ("tech support slow", "General Inquiry"),
    ("dont understand invoice", "Billing"),
    ("logins not working again", "Technical Support"),
    ("who handles refunds", "Billing")
]

def create_ticket_dataset():
    tickets = [
        ("I can't log into my account", "Technical Support"),
        ("How do I update my payment method?", "Billing"),
        ("The app crashes on startup", "Technical Support"),
        ("I'd like to close my account", "Account Management"),
        ("I was charged twice this month", "Billing"),
        ("What’s your refund policy?", "General Inquiry"),
        ("Can I change my subscription plan?", "Account Management"),
        ("Why was my account suspended?", "Account Management"),
        ("The system keeps logging me out", "Technical Support"),
        ("Where do I download my invoices?", "Billing"),
        ("I can’t access the app after payment", "Billing"), # mislabeled or ambiguous examples
        ("How do I reset my router?", "Billing"), # mislabeled or ambiguous examples
        ("I need help with billing and login", "Technical Support"), # mislabeled or ambiguous examples
        ("What plans do you offer for families?", "General Inquiry"), # mislabeled or ambiguous examples
        ("I want to cancel and get a refund", "Account Management"), # mislabeled or ambiguous examples
        ("My payment failed on the last transaction", "Billing"),
        ("Where can I find the user guide?", "General Inquiry"),
        ("Reset password link is not working", "Technical Support"),
        ("I received an incorrect invoice", "Billing"),
        ("How do I remove a user from our team account?", "Account Management"),
        ("My account was upgraded without consent", "Account Management"),
        ("Can I get a yearly subscription discount?", "General Inquiry"),
        ("App takes too long to load", "Technical Support"),
        ("I want to export all billing history", "Billing"),
        ("System is down, when will it be restored?", "Technical Support"),
        ("How do I delete my saved payment methods?", "Billing"),
        ("Is there a plan suitable for small businesses?", "General Inquiry"),
        ("Getting 500 error on dashboard login", "Technical Support"),
        ("Need to merge two team accounts", "Account Management"),
        ("Do you offer phone support?", "General Inquiry"),
        ("Unable to apply promo code at checkout", "Billing"),
        ("Is my data backed up regularly?", "General Inquiry"),
        ("Transferred money but not showing in balance", "Billing"),
        ("App is showing incorrect data after last update", "Technical Support"),
        ("Can I add multiple admins to my account?", "Account Management"),
        ("I want to upgrade my account to premium", "Billing"),  # mislabeled or ambiguous examples
        ("My password reset email never arrived", "Billing"),  # mislabeled or ambiguous examples
        ("Cancel my subscription ASAP", "Technical Support"),  # mislabeled or ambiguous examples
        ("Is my invoice ready for review?", "General Inquiry"),  # mislabeled or ambiguous examples
        ("Can I call someone to fix the bug?", "Billing"),  # mislabeled or ambiguous examples
        ("I need to deactivate one of our users", "Account Management"),
        ("Can I downgrade from Pro to Basic?", "Account Management"),
        ("My admin privileges were removed", "Account Management"),
        ("Transfer account ownership to another team member", "Account Management"),
        ("Update company address in billing", "Account Management"),
        ("Add more seats to my current plan", "Account Management"),
        ("Set up SSO for all users", "Account Management"),
        ("Change the team name on our dashboard", "Account Management"),
        ("I’m locked out of the admin panel", "Account Management"),
        ("Give view-only access to some users", "Account Management"),
        ("Can I see a demo of your product?", "General Inquiry"),
        ("Where can I find your privacy policy?", "General Inquiry"),
        ("Do you offer educational discounts?", "General Inquiry"),
        ("Is there a help center or FAQ?", "General Inquiry"),
        ("How does your free trial work?", "General Inquiry"),
        ("What languages does the platform support?", "General Inquiry"),
        ("Are there any hidden fees?", "General Inquiry"),
        ("Can I schedule a call to learn more?", "General Inquiry"),
        ("What is the uptime guarantee?", "General Inquiry"),
        ("Where do I manage notifications?", "General Inquiry"),

    ] + noisy_samples   # replicate for size
    df = pd.DataFrame(tickets, columns=['text', 'label'])
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

raw_df = create_ticket_dataset()
raw_df.head()

In [ ]:
model_name = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(model_name)

X = embedder.encode(raw_df['text'].tolist())
y = raw_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
data_version = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

metadata = {
    "embedding_model": model_name,
    "data_version": data_version,
    "classes": sorted(raw_df['label'].unique().tolist()),
    "created": datetime.datetime.now().isoformat()
}
with open(f"embedding_metadata_{data_version}.json", "w") as f:
    json.dump(metadata, f, indent=2)

## Step 2: Experimentation & Training

In [ ]:
# Create models and train
models = {
    "LogisticRegression": LogisticRegression(max_iter=500),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM (Linear Kernel)": SVC(kernel='linear', probability=True, random_state=42)
}

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# The ConfusionMatrixDisplay and plotting should be inside the loop
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test) # y_pred is calculated here for each model
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        mlflow.log_param("model_type", name)
        mlflow.log_param("embedding_model", model_name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        print(f"\nModel: {name}")
        print(classification_report(y_test, y_pred))

        # Now plot the Confusion Matrix for the current model
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=45)
        plt.title(f"Confusion Matrix: {name}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Collect metrics for manual comparison
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    })

results_df = pd.DataFrame(results)
results_df.plot(x='Model', kind='bar', legend=True, figsize=(8,5), title='Model Comparison')
plt.xticks(rotation=45)
plt.ylabel("Score")
plt.tight_layout()
plt.show()

## Step 3: Automated Validation (CI)

In [ ]:
def test_embedding_output(X):
    assert X.shape[1] == 384, "Embedding dimensionality mismatch."
    print("Test passed: Embedding output shape correct.")

def test_class_distribution(y):
    counts = pd.Series(y).value_counts()
    assert all(counts > 10), "Class imbalance too severe."
    print("Test passed: Class distribution sufficient.")

# Run tests
print("Running validation tests...")
test_embedding_output(X_train)
test_class_distribution(y_train)

## Step 4: Packaging & Continuous Delivery (CD)

In [ ]:
# Select best model from Lab 1
best_model_name = results_df.sort_values("F1 Score", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Selected best model: {best_model_name}")

# Package model
import pickle
import shutil
from pathlib import Path

version_id = f"v_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
package_path = Path("deployment") / version_id
package_path.mkdir(parents=True, exist_ok=True)

with open(package_path / "model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open(package_path / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

example_input = raw_df.iloc[:2].to_dict(orient="records")
with open(package_path / "example_input.json", "w") as f:
    json.dump(example_input, f, indent=2)

print(f"Model packaged at: {package_path}")


## Step 5: Production Monitoring & Observability

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()
        df = pd.DataFrame(data)
        X_new = embedder.encode(df['text'].tolist())
        preds = best_model.predict(X_new)
        return jsonify({"predictions": preds.tolist()})
    except Exception as e:
        return jsonify({"error": str(e)}), 400

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model_version": version_id})

# Simulated usage: Drift/traffic alert
import random
fake_preds = [random.choice(["Billing", "Technical Support", "Account Management", "General Inquiry"]) for _ in range(100)]
pd.Series(fake_preds).value_counts().plot(kind="bar", title="Prediction Volume by Class")
plt.show()


## Step 6: Governance & Continuous Improvement

In [ ]:
model_card = {
    "model_name": "Ticket Routing Classifier",
    "version": version_id,
    "created": metadata["created"],
    "data_version": metadata["data_version"],
    "description": "Routes customer tickets using LLM embeddings and classification.",
    "performance_summary": results_df.set_index("Model").to_dict(orient="index"),
    "ethical_considerations": [
        "Bias may arise from class imbalance in support ticket data.",
        "Model should not be used for disciplinary action."
    ],
    "retraining_criteria": "Re-evaluate every 30 days or if accuracy drops >10%."
}

with open(package_path / "model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)

print("Model Card Summary:")
print(json.dumps(model_card["performance_summary"], indent=2))
